In [68]:
import re
import requests
import geocoder
import time

In [69]:
url = 'http://127.0.0.1:1234/v1/chat/completions'

In [70]:
items = []

print('')

while True:
    item = input("welke gerechten heb je al/ heb je die je wilt gebruiken. (type stop om te stoppen): ")

    if item == "stop":
        break

    items.append(item)

In [71]:
g = geocoder.ip("me")
regio = g.city
lat = g.latlng[0]
lon = g.latlng[1]

In [72]:
prompts = [
    f"""
Geef exact 5 gerechten op basis van deze ingrediënten: {items}.

vaste opzet:
gerecht nr|gerecht|ingredienten|duur|nieuwe ingredienten|

Voor elk gerecht:
- bereidingstijd <= 30 minuten
- extra ingrediënt
- en zet letterlijk dit aan het einde van de text zodat ik hem er later uit kan halen: 'nieuw ingredient = (ingredient)'
"""
]

In [73]:
for prompt in prompts:
    start_time = time.time()

In [74]:
data = {
    "model": "nvidia/nemotron-3-nano-4b",
    "messages": [
        {"role": "user", "content": prompt}
    ],
    "temperature": 0.7
}


In [75]:
try:
    response = requests.post(url, json=data)
    response.raise_for_status()

    end_time = time.time()

    result = response.json()

    answer = result["choices"][0]["message"]["content"]
    model_name = result.get("model", "onbekend")
    response_length = len(answer)

    print(answer)

    print("\nMetadata:")
    print(f"Modelnaam: {model_name}")
    print(f"Lengte response: {response_length} karakters")
    print(f"Tijd: {end_time - start_time:.2f} seconden")
    print("-" * 50)

except requests.exceptions.RequestException as e:
    print(f"Fout bij API-aanroep: {e}")


1|Andijvi met hamplakjes en tomaatensaus (noodels, aardappelen)|noedels, aardappelen, hamplakjes, tomaatensoep|25 min|nieuw ingrediënt = (ui)  
2|Biefstuk gehakt met tomatensoep en melkcreme|biefstuk, gehakt, tomaatensoep, melk|30 min|nieuw ingrediënt = (beteram)  
3|Noodels in tomatensoep met aardappelpurée|noedels, aardappelen, tomaatensoep|25 min|nieuw ingrediënt = (paprika)  
4|Hamplakjes op aardappelen met melk|hamplakjes, aardappelen, melk|20 min|nieuw ingrediënt = (mosterd)  
5|Biefstuk met tomatensoep|biefstuk, tomaatensoep|20 min|nieuw ingrediënt = (wortelkool)

Metadata:
Modelnaam: nvidia/nemotron-3-nano-4b
Lengte response: 577 karakters
Tijd: 30.50 seconden
--------------------------------------------------


In [79]:
for answe in answer.splitlines():
    match = re.search(r"nieuw ingrediënt\s*=\s*\((.*?)\)", answe)

    if match:
        ingredient = match.group(1)
        url = f"https://google.com/search?q=supermarkt+voor+{ingredient}+-ai"
        answe += f"|{url}"

    print(answe)



1|Andijvi met hamplakjes en tomaatensaus (noodels, aardappelen)|noedels, aardappelen, hamplakjes, tomaatensoep|25 min|nieuw ingrediënt = (ui)  |https://google.com/search?q=supermarkt+voor+ui+-ai
2|Biefstuk gehakt met tomatensoep en melkcreme|biefstuk, gehakt, tomaatensoep, melk|30 min|nieuw ingrediënt = (beteram)  |https://google.com/search?q=supermarkt+voor+beteram+-ai
3|Noodels in tomatensoep met aardappelpurée|noedels, aardappelen, tomaatensoep|25 min|nieuw ingrediënt = (paprika)  |https://google.com/search?q=supermarkt+voor+paprika+-ai
4|Hamplakjes op aardappelen met melk|hamplakjes, aardappelen, melk|20 min|nieuw ingrediënt = (mosterd)  |https://google.com/search?q=supermarkt+voor+mosterd+-ai
5|Biefstuk met tomatensoep|biefstuk, tomaatensoep|20 min|nieuw ingrediënt = (wortelkool)|https://google.com/search?q=supermarkt+voor+wortelkool+-ai
